In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/f1_historical_results_2021_2025.csv")

df.head()

,season,round,event,driver,driver_name,team,grid_position,race_position,race_points,winner,podium,qualifying_position
0,2021,1,Bahrain Grand Prix,HAM,Lewis Hamilton,Mercedes,2.0,1.0,25.0,1,1,2.0
1,2021,1,Bahrain Grand Prix,VER,Max Verstappen,Red Bull Racing,1.0,2.0,18.0,0,1,1.0
2,2021,1,Bahrain Grand Prix,BOT,Valtteri Bottas,Mercedes,3.0,3.0,16.0,0,1,3.0
3,2021,1,Bahrain Grand Prix,NOR,Lando Norris,McLaren,7.0,4.0,12.0,0,0,7.0
4,2021,1,Bahrain Grand Prix,PER,Sergio Perez,Red Bull Racing,0.0,5.0,10.0,0,0,11.0


In [40]:
df.shape
df.columns.tolist()

['season',
 'round',
 'event',
 'driver',
 'driver_name',
 'team',
 'grid_position',
 'race_position',
 'race_points',
 'winner',
 'podium',
 'qualifying_position']

In [41]:
df = df.sort_values(['season', 'round'])

df.head(10)

,season,round,event,driver,driver_name,team,grid_position,race_position,race_points,winner,podium,qualifying_position
0,2021,1,Bahrain Grand Prix,HAM,Lewis Hamilton,Mercedes,2.0,1.0,25.0,1,1,2.0
1,2021,1,Bahrain Grand Prix,VER,Max Verstappen,Red Bull Racing,1.0,2.0,18.0,0,1,1.0
2,2021,1,Bahrain Grand Prix,BOT,Valtteri Bottas,Mercedes,3.0,3.0,16.0,0,1,3.0
3,2021,1,Bahrain Grand Prix,NOR,Lando Norris,McLaren,7.0,4.0,12.0,0,0,7.0
4,2021,1,Bahrain Grand Prix,PER,Sergio Perez,Red Bull Racing,0.0,5.0,10.0,0,0,11.0
5,2021,1,Bahrain Grand Prix,LEC,Charles Leclerc,Ferrari,4.0,6.0,8.0,0,0,4.0
6,2021,1,Bahrain Grand Prix,RIC,Daniel Ricciardo,McLaren,6.0,7.0,6.0,0,0,6.0
7,2021,1,Bahrain Grand Prix,SAI,Carlos Sainz,Ferrari,8.0,8.0,4.0,0,0,8.0
8,2021,1,Bahrain Grand Prix,TSU,Yuki Tsunoda,AlphaTauri,13.0,9.0,2.0,0,0,13.0
9,2021,1,Bahrain Grand Prix,STR,Lance Stroll,Aston Martin,10.0,10.0,1.0,0,0,10.0


In [42]:
df[['season', 'round', 'driver_name', 'grid_position',
   'qualifying_position', 'race_position', 'winner', 'podium']].head(20)

,season,round,driver_name,grid_position,qualifying_position,race_position,winner,podium
0,2021,1,Lewis Hamilton,2.0,2.0,1.0,1,1
1,2021,1,Max Verstappen,1.0,1.0,2.0,0,1
2,2021,1,Valtteri Bottas,3.0,3.0,3.0,0,1
3,2021,1,Lando Norris,7.0,7.0,4.0,0,0
4,2021,1,Sergio Perez,0.0,11.0,5.0,0,0
5,2021,1,Charles Leclerc,4.0,4.0,6.0,0,0
6,2021,1,Daniel Ricciardo,6.0,6.0,7.0,0,0
7,2021,1,Carlos Sainz,8.0,8.0,8.0,0,0
8,2021,1,Yuki Tsunoda,13.0,13.0,9.0,0,0
9,2021,1,Lance Stroll,10.0,10.0,10.0,0,0


In [43]:
# Previous finish within the same season
df['previous_finish'] = (
    df.groupby(['season', 'driver'])['race_position']
      .shift(1)
)

# Average finishing position of previous 3 races
df['recent_avg_finish'] = (
    df.groupby(['season', 'driver'])['race_position']
      .transform(
          lambda x: x.shift(1).rolling(3, min_periods=1).mean()
      )
)

In [44]:
# Driver championship points before the current race
df['driver_points_before'] = (
    df.groupby('driver')['race_points']
      .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

In [45]:
# Constructor championship points before the current race
df['constructor_points_before'] = (
    df.groupby(['season', 'team'])['race_points']
      .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

# Should print [0.] if the fixed feature reached the model
print(test_df[test_df['round'] == 1]['constructor_points_before'].unique())
print(X_train[train_df['round'] == 1]['constructor_points_before'].unique())

[ 0. 15. 25.  4. 18.  6.  8. 10.]
[ 0. 25. 18. 12.  8.  1.  2. 26. 15.  6. 10.  4.]


In [46]:
# Check your new dataset
df.head(20)

,season,round,event,driver,driver_name,team,grid_position,race_position,race_points,winner,podium,qualifying_position,previous_finish,recent_avg_finish,driver_points_before,constructor_points_before
0,2021,1,Bahrain Grand Prix,HAM,Lewis Hamilton,Mercedes,2.0,1.0,25.0,1,1,2.0,NaN,NaN,0.0,0.0
1,2021,1,Bahrain Grand Prix,VER,Max Verstappen,Red Bull Racing,1.0,2.0,18.0,0,1,1.0,NaN,NaN,0.0,0.0
2,2021,1,Bahrain Grand Prix,BOT,Valtteri Bottas,Mercedes,3.0,3.0,16.0,0,1,3.0,NaN,NaN,0.0,25.0
3,2021,1,Bahrain Grand Prix,NOR,Lando Norris,McLaren,7.0,4.0,12.0,0,0,7.0,NaN,NaN,0.0,0.0
4,2021,1,Bahrain Grand Prix,PER,Sergio Perez,Red Bull Racing,0.0,5.0,10.0,0,0,11.0,NaN,NaN,0.0,18.0
5,2021,1,Bahrain Grand Prix,LEC,Charles Leclerc,Ferrari,4.0,6.0,8.0,0,0,4.0,NaN,NaN,0.0,0.0
6,2021,1,Bahrain Grand Prix,RIC,Daniel Ricciardo,McLaren,6.0,7.0,6.0,0,0,6.0,NaN,NaN,0.0,12.0
7,2021,1,Bahrain Grand Prix,SAI,Carlos Sainz,Ferrari,8.0,8.0,4.0,0,0,8.0,NaN,NaN,0.0,8.0
8,2021,1,Bahrain Grand Prix,TSU,Yuki Tsunoda,AlphaTauri,13.0,9.0,2.0,0,0,13.0,NaN,NaN,0.0,0.0
9,2021,1,Bahrain Grand Prix,STR,Lance Stroll,Aston Martin,10.0,10.0,1.0,0,0,10.0,NaN,NaN,0.0,0.0


In [47]:
df.columns.tolist()

['season',
 'round',
 'event',
 'driver',
 'driver_name',
 'team',
 'grid_position',
 'race_position',
 'race_points',
 'winner',
 'podium',
 'qualifying_position',
 'previous_finish',
 'recent_avg_finish',
 'driver_points_before',
 'constructor_points_before']

In [48]:
# verify the points calculation and chronological ordering, especially because your dataset covers multiple seasons
# confirm that the model is only receiving information that existed before each race, avoiding data leakage
df[df['driver'] == df['driver'].iloc[0]][
    ['season', 'round', 'driver_name', 'race_position',
     'race_points', 'driver_points_before', 'previous_finish',
     'recent_avg_finish']
].head(10)

,season,round,driver_name,race_position,race_points,driver_points_before,previous_finish,recent_avg_finish
0,2021,1,Lewis Hamilton,1.0,25.0,0.0,NaN,NaN
21,2021,2,Lewis Hamilton,2.0,19.0,25.0,1.0,1.000000
40,2021,3,Lewis Hamilton,1.0,25.0,44.0,2.0,1.500000
60,2021,4,Lewis Hamilton,1.0,25.0,69.0,1.0,1.333333
86,2021,5,Lewis Hamilton,7.0,7.0,94.0,1.0,1.333333
114,2021,6,Lewis Hamilton,15.0,0.0,101.0,7.0,3.000000
121,2021,7,Lewis Hamilton,2.0,18.0,101.0,15.0,7.666667
141,2021,8,Lewis Hamilton,2.0,19.0,119.0,2.0,8.000000
163,2021,9,Lewis Hamilton,4.0,12.0,138.0,2.0,6.333333
180,2021,10,Lewis Hamilton,1.0,25.0,150.0,4.0,2.666667


In [49]:
# Check missing values in each column
df.isnull().sum()

season                         0
round                          0
event                          0
driver                         0
driver_name                    0
team                           0
grid_position                  3
race_position                  3
race_points                    0
winner                         0
podium                         0
qualifying_position            2
previous_finish              112
recent_avg_finish            110
driver_points_before           0
constructor_points_before      0
dtype: int64

In [50]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [51]:
features = [
    'grid_position',
    'qualifying_position',
    'driver_points_before',
    'constructor_points_before',
    'previous_finish',
    'recent_avg_finish'
]

X = df[features]

# Targets
y_winner = df['winner']
y_podium = df['podium']

In [52]:
# check the result
print("Features:", X.shape)
print("Winner target:", y_winner.shape)
print("Podium target:", y_podium.shape)

print("\nMissing values:")
print(X.isnull().sum())

Features: (2278, 6)
Winner target: (2278,)
Podium target: (2278,)

Missing values:
grid_position                  3
qualifying_position            2
driver_points_before           0
constructor_points_before      0
previous_finish              112
recent_avg_finish            110
dtype: int64


In [53]:
# Indicate whether a driver has previous race history
df['no_previous_finish'] = df['previous_finish'].isnull().astype(int)

df['no_recent_history'] = df['recent_avg_finish'].isnull().astype(int)

In [54]:
# Fill missing grid and qualifying positions
df['grid_position'] = df['grid_position'].fillna(
    df['grid_position'].median()
)

df['qualifying_position'] = df['qualifying_position'].fillna(
    df['qualifying_position'].median()
)

# Fill missing historical features
df['previous_finish'] = df['previous_finish'].fillna(0)

df['recent_avg_finish'] = df['recent_avg_finish'].fillna(0)

In [55]:
print(df[features].isnull().sum())

grid_position                0
qualifying_position          0
driver_points_before         0
constructor_points_before    0
previous_finish              0
recent_avg_finish            0
dtype: int64


In [56]:
print(df[['no_previous_finish', 'no_recent_history']].sum())

no_previous_finish    112
no_recent_history     110
dtype: int64


In [57]:
# Check available seasons
print(sorted(df['season'].unique()))

[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [58]:
# Define features
features = [
    'grid_position',
    'qualifying_position',
    'driver_points_before',
    'constructor_points_before',
    'previous_finish',
    'recent_avg_finish',
    'no_previous_finish',
    'no_recent_history'
]

# Training data: 2021–2024
train_df = df[df['season'] <= 2024].copy()

# Testing data: 2025
test_df = df[df['season'] == 2025].copy()

# Create input features
X_train = train_df[features]
X_test = test_df[features]

# Winner targets
y_train_winner = train_df['winner']
y_test_winner = test_df['winner']

# Podium targets
y_train_podium = train_df['podium']
y_test_podium = test_df['podium']

# Check dataset sizes
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training seasons:", sorted(train_df['season'].unique()))
print("Testing seasons:", sorted(test_df['season'].unique()))

Training data: (1799, 8)
Testing data: (479, 8)
Training seasons: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Testing seasons: [np.int64(2025)]


In [59]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
import sklearn
print(sklearn.__version__)

1.9.1


In [61]:
from sklearn.ensemble import RandomForestClassifier

# 1. Create winner prediction model
winner_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

winner_model.fit(X_train, y_train_winner)


# 2. Create podium prediction model
podium_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

podium_model.fit(X_train, y_train_podium)


print("Winner model trained successfully!")
print("Podium model trained successfully!")

Winner model trained successfully!
Podium model trained successfully!


In [62]:

# winner model prediction
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Generate winner predictions
y_pred_winner = winner_model.predict(X_test)

# Evaluate Winner Model
print("===== WINNER MODEL =====")

print("Accuracy:", accuracy_score(
    y_test_winner, y_pred_winner
))

print("\nClassification Report:")
print(classification_report(
    y_test_winner,
    y_pred_winner,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_winner, y_pred_winner
))

===== WINNER MODEL =====
Accuracy: 0.9352818371607515

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.97       455
           1       0.40      0.58      0.47        24

    accuracy                           0.94       479
   macro avg       0.69      0.77      0.72       479
weighted avg       0.95      0.94      0.94       479


Confusion Matrix:
[[434  21]
 [ 10  14]]


In [63]:
# podium model prediction
# Generate podium predictions
y_pred_podium = podium_model.predict(X_test)

# Evaluate Podium Model
print("===== PODIUM MODEL =====")

print("Accuracy:", accuracy_score(
    y_test_podium, y_pred_podium
))

print("\nClassification Report:")
print(classification_report(
    y_test_podium,
    y_pred_podium,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_podium, y_pred_podium
))

===== PODIUM MODEL =====
Accuracy: 0.906054279749478

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94       407
           1       0.65      0.82      0.72        72

    accuracy                           0.91       479
   macro avg       0.81      0.87      0.83       479
weighted avg       0.92      0.91      0.91       479


Confusion Matrix:
[[375  32]
 [ 13  59]]


In [64]:
# Get winner prediction probabilities
winner_probabilities = winner_model.predict_proba(X_test)[:, 1]

print("Winner probabilities generated successfully!")
print(winner_probabilities[:10])

Winner probabilities generated successfully!
[1.   0.21 0.04 0.   0.01 0.   0.   0.12 0.16 0.17]


In [65]:
# Get podium prediction probabilities
podium_probabilities = podium_model.predict_proba(X_test)[:, 1]

print("Podium probabilities generated successfully!")
print(podium_probabilities[:10])

Podium probabilities generated successfully!
[0.99 0.41 0.94 0.   0.11 0.   0.   0.14 0.88 0.16]


In [66]:
# Put the probabilities together with the drivers
prediction_results = test_df[
    ['season', 'round', 'event', 'driver_name', 'team',
     'qualifying_position', 'grid_position',
     'race_position', 'winner', 'podium']
].copy()

prediction_results['winner_probability'] = winner_probabilities
prediction_results['podium_probability'] = podium_probabilities

prediction_results.head(20)

,season,round,event,driver_name,team,qualifying_position,grid_position,race_position,winner,podium,winner_probability,podium_probability
1799,2025,1,Australian Grand Prix,Lando Norris,McLaren,1.0,1.0,1.0,1,1,1.00,0.99
1800,2025,1,Australian Grand Prix,Max Verstappen,Red Bull Racing,3.0,3.0,2.0,0,1,0.21,0.41
1801,2025,1,Australian Grand Prix,George Russell,Mercedes,4.0,4.0,3.0,0,1,0.04,0.94
1802,2025,1,Australian Grand Prix,Andrea Kimi Antonelli,Mercedes,16.0,16.0,4.0,0,0,0.00,0.00
1803,2025,1,Australian Grand Prix,Alexander Albon,Williams,6.0,6.0,5.0,0,0,0.01,0.11
1804,2025,1,Australian Grand Prix,Lance Stroll,Aston Martin,13.0,13.0,6.0,0,0,0.00,0.00
1805,2025,1,Australian Grand Prix,Nico Hulkenberg,Kick Sauber,17.0,17.0,7.0,0,0,0.00,0.00
1806,2025,1,Australian Grand Prix,Charles Leclerc,Ferrari,7.0,7.0,8.0,0,0,0.12,0.14
1807,2025,1,Australian Grand Prix,Oscar Piastri,McLaren,2.0,2.0,9.0,0,0,0.16,0.88
1808,2025,1,Australian Grand Prix,Lewis Hamilton,Ferrari,8.0,8.0,10.0,0,0,0.17,0.16


In [67]:
# Sort the results by winner probability and display the top 20 predictions
prediction_results.sort_values(
    'winner_probability',
    ascending=False
)[[
    'season',
    'round',
    'event',
    'driver_name',
    'race_position',
    'winner',
    'winner_probability'
]].head(20)

,season,round,event,driver_name,race_position,winner,winner_probability
2236,2025,22,Las Vegas Grand Prix,Lando Norris,19.0,0,1.00
2198,2025,21,São Paulo Grand Prix,Lando Norris,1.0,1,1.00
1799,2025,1,Australian Grand Prix,Lando Norris,1.0,1,1.00
1921,2025,7,Emilia Romagna Grand Prix,Oscar Piastri,3.0,0,0.89
2095,2025,15,Dutch Grand Prix,Lando Norris,18.0,0,0.88
2078,2025,15,Dutch Grand Prix,Oscar Piastri,1.0,1,0.86
2039,2025,13,Belgian Grand Prix,Lando Norris,2.0,0,0.86
1998,2025,11,Austrian Grand Prix,Lando Norris,1.0,1,0.85
2178,2025,20,Mexico City Grand Prix,Lando Norris,1.0,1,0.83
1960,2025,9,Spanish Grand Prix,Lando Norris,2.0,0,0.78


In [68]:

# podium highest probability
prediction_results.sort_values(
    'podium_probability',
    ascending=False
)[[
    'season',
    'round',
    'event',
    'driver_name',
    'race_position',
    'podium',
    'podium_probability'
]].head(20)

,season,round,event,driver_name,race_position,podium,podium_probability
2236,2025,22,Las Vegas Grand Prix,Lando Norris,19.0,0,1.00
1821,2025,2,Chinese Grand Prix,George Russell,3.0,1,1.00
2198,2025,21,São Paulo Grand Prix,Lando Norris,1.0,1,1.00
2095,2025,15,Dutch Grand Prix,Lando Norris,18.0,0,1.00
1799,2025,1,Australian Grand Prix,Lando Norris,1.0,1,0.99
1900,2025,6,Miami Grand Prix,Lando Norris,2.0,1,0.99
1840,2025,3,Japanese Grand Prix,Lando Norris,2.0,1,0.98
2000,2025,11,Austrian Grand Prix,Charles Leclerc,3.0,1,0.97
1883,2025,5,Saudi Arabian Grand Prix,George Russell,5.0,0,0.97
1921,2025,7,Emilia Romagna Grand Prix,Oscar Piastri,3.0,1,0.96


In [69]:
# Group test rows by race
races = prediction_results.groupby(['season', 'round'])

# Sanity check: number of races and winners per race
print("Number of races:", races.ngroups)
print("Winners per race:", races['winner'].sum().value_counts().to_dict())

# Rank drivers within each race by predicted winner probability
prediction_results['winner_rank'] = races['winner_probability'].rank(
    method='first', ascending=False
)

# Top-ranked driver per race
top_winner = prediction_results[prediction_results['winner_rank'] == 1]

print("Top-1 winner hit rate:", top_winner['winner'].mean())
print("Correct winner predictions:", int(top_winner['winner'].sum()), "/", len(top_winner))

# Model's pick vs actual result for each race
top_winner[['round', 'event', 'driver_name', 'winner_probability', 'race_position']]

Number of races: 24
Winners per race: {1: 24}
Top-1 winner hit rate: 0.375
Correct winner predictions: 9 / 24


,round,event,driver_name,winner_probability,race_position
1799,1,Australian Grand Prix,Lando Norris,1.00,1.0
1821,2,Chinese Grand Prix,George Russell,0.45,3.0
1840,3,Japanese Grand Prix,Lando Norris,0.54,2.0
1861,4,Bahrain Grand Prix,Lando Norris,0.47,3.0
1880,5,Saudi Arabian Grand Prix,Max Verstappen,0.55,2.0
1900,6,Miami Grand Prix,Lando Norris,0.71,2.0
1921,7,Emilia Romagna Grand Prix,Oscar Piastri,0.89,3.0
1941,8,Monaco Grand Prix,Oscar Piastri,0.70,3.0
1960,9,Spanish Grand Prix,Lando Norris,0.78,2.0
1978,10,Canadian Grand Prix,George Russell,0.77,1.0


In [70]:
# Full race-by-race winner ranking
winner_rankings = prediction_results.sort_values(
    ['season', 'round', 'winner_rank']
)

winner_rankings[
    ['round', 'event', 'driver_name', 'winner_probability',
     'winner_rank', 'race_position', 'winner']
].head(40)

,round,event,driver_name,winner_probability,winner_rank,race_position,winner
1799,1,Australian Grand Prix,Lando Norris,1.00,1.0,1.0,1
1800,1,Australian Grand Prix,Max Verstappen,0.21,2.0,2.0,0
1808,1,Australian Grand Prix,Lewis Hamilton,0.17,3.0,10.0,0
1807,1,Australian Grand Prix,Oscar Piastri,0.16,4.0,9.0,0
1806,1,Australian Grand Prix,Charles Leclerc,0.12,5.0,8.0,0
1816,1,Australian Grand Prix,Carlos Sainz,0.05,6.0,18.0,0
1801,1,Australian Grand Prix,George Russell,0.04,7.0,3.0,0
1803,1,Australian Grand Prix,Alexander Albon,0.01,8.0,5.0,0
1802,1,Australian Grand Prix,Andrea Kimi Antonelli,0.00,9.0,4.0,0
1804,1,Australian Grand Prix,Lance Stroll,0.00,10.0,6.0,0


In [71]:
winner_rankings[winner_rankings['round'] == 22][
    ['event', 'driver_name', 'winner_probability', 'winner_rank', 'race_position']
]

,event,driver_name,winner_probability,winner_rank,race_position
2236,Las Vegas Grand Prix,Lando Norris,1.00,1.0,19.0
2218,Las Vegas Grand Prix,Max Verstappen,0.29,2.0,1.0
2219,Las Vegas Grand Prix,George Russell,0.29,3.0,2.0
2222,Las Vegas Grand Prix,Carlos Sainz,0.08,4.0,5.0
2221,Las Vegas Grand Prix,Charles Leclerc,0.04,5.0,4.0
2220,Las Vegas Grand Prix,Kimi Antonelli,0.03,6.0,3.0
2229,Las Vegas Grand Prix,Yuki Tsunoda,0.02,7.0,12.0
2223,Las Vegas Grand Prix,Isack Hadjar,0.01,8.0,6.0
2231,Las Vegas Grand Prix,Liam Lawson,0.01,9.0,14.0
2237,Las Vegas Grand Prix,Oscar Piastri,0.01,10.0,20.0


In [72]:
# Rank drivers within each race by predicted podium probability
prediction_results['podium_rank'] = (
    prediction_results.groupby(['season', 'round'])['podium_probability']
    .rank(method='first', ascending=False)
)

# Top 3 predicted drivers per race
top3 = prediction_results[prediction_results['podium_rank'] <= 3]

print("Top-3 podium precision:", top3['podium'].mean())
print("Correct podium selections:", int(top3['podium'].sum()), "/", len(top3))

Top-3 podium precision: 0.7361111111111112
Correct podium selections: 53 / 72


In [73]:
# Constructor points before each race (earlier rounds only, no current-race leakage)
team_round = (
    df.groupby(['season', 'team', 'round'])['race_points']
      .sum()
      .reset_index()
      .sort_values(['season', 'team', 'round'])
)

team_round['constructor_points_before'] = (
    team_round.groupby(['season', 'team'])['race_points']
      .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

df = df.drop(columns=['constructor_points_before'], errors='ignore').merge(
    team_round[['season', 'team', 'round', 'constructor_points_before']],
    on=['season', 'team', 'round'],
    how='left'
)

# Quick check: first race of each season should be 0
print(df[df['round'] == 1]['constructor_points_before'].unique())

[0.]


In [74]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Winner model
winner_pred = winner_model.predict(X_test)
print("WINNER MODEL")
print("Accuracy:", accuracy_score(y_test_winner, winner_pred))
print("Confusion matrix:\n", confusion_matrix(y_test_winner, winner_pred))

# Podium model
podium_pred = podium_model.predict(X_test)
print("\nPODIUM MODEL")
print("Accuracy:", accuracy_score(y_test_podium, podium_pred))
print("Confusion matrix:\n", confusion_matrix(y_test_podium, podium_pred))

# Race-level metrics
prediction_results['winner_probability'] = winner_model.predict_proba(X_test)[:, 1]
prediction_results['podium_probability'] = podium_model.predict_proba(X_test)[:, 1]

races = prediction_results.groupby(['season', 'round'])
prediction_results['winner_rank'] = races['winner_probability'].rank(method='first', ascending=False)
prediction_results['podium_rank'] = races['podium_probability'].rank(method='first', ascending=False)

top1 = prediction_results[prediction_results['winner_rank'] == 1]
top3 = prediction_results[prediction_results['podium_rank'] <= 3]

print("\nTop-1 winner hit rate:", int(top1['winner'].sum()), "/", len(top1))
print("Top-3 podium precision:", int(top3['podium'].sum()), "/", len(top3))

WINNER MODEL
Accuracy: 0.9352818371607515
Confusion matrix:
 [[434  21]
 [ 10  14]]

PODIUM MODEL
Accuracy: 0.906054279749478
Confusion matrix:
 [[375  32]
 [ 13  59]]

Top-1 winner hit rate: 9 / 24
Top-3 podium precision: 53 / 72
